# Etapa 2 — Preparar a estrutura do projeto
Use a pasta Git do Databricks, na branch `chore/estrutura-inicial`.

Esta etapa importa o pacote Python, valida parâmetros e cria/verifica cinco schemas dev. Não cria tabelas nem volumes. O path mostrado é planejado; acesso a arquivos será validado na próxima etapa. Não é necessário instalar bibliotecas.

**Parâmetros:** execute apenas a primeira célula de código para exibir os widgets. Preencha os campos e depois execute as demais células. Use **Run all** quando os parâmetros já estiverem preenchidos.


In [0]:
# catalogo: workspace (obrigatório). Destino dos cinco schemas de desenvolvimento.
# modo: incremental nesta validação. O bootstrap valida a configuração; não ingere dados.
# data_referencia: deixe vazio nesta rodada. Em reprocessamento, informe a data alvo YYYY-MM-DD.
# batch_id: deixe vazio para gerar UUID. Preencha apenas para identificar explicitamente um lote.
# raiz_arquivos: deixe vazio para usar /Volumes/<catalogo>/medallion_dev_landing/arquivos.
# Um caminho explícito deve ser /Volumes/...; aqui ele é apenas planejado, sem validar acesso.
# Execute somente este CMD, preencha os widgets e então prossiga.
# 1. Exibir parâmetros — execute esta célula antes de preencher os campos.
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
dbutils.widgets.dropdown("modo", "incremental", ["incremental", "reprocessamento"])
dbutils.widgets.text("data_referencia", "")
dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("raiz_arquivos", "")

print("Preencha os widgets acima e depois execute as próximas células.")


In [0]:
# CMD 2 — Localizar a pasta Git e importar o pacote. Nenhum preenchimento adicional; execute dentro do repositório.
from pathlib import Path
import sys

atual = Path.cwd().resolve()
candidatos = [atual, *atual.parents]
raiz = next(
    (p for p in candidatos if (p / "src" / "medallion" / "__init__.py").is_file()
     and (p / "pyproject.toml").is_file()),
    None,
)
if raiz is None:
    raise RuntimeError(
        "Não encontrei a raiz do repositório. Abra este notebook dentro da pasta Git "
        "e confirme que src/medallion e pyproject.toml vieram no Pull. "
        "Se necessário, use %cd com o caminho da pasta Git e execute novamente."
    )
src = str(raiz / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from medallion.config import PipelineConfig
from medallion.bootstrap import preparar_ambiente

print("Pacote medallion importado de:", raiz)



In [0]:
# CMD 3 — Ler os widgets preenchidos e validar a configuração. Conferir catálogo, batch e caminho planejado no output.

parametros = {
    "ambiente": "dev",
    "catalogo": dbutils.widgets.get("catalogo").strip(),
    "modo": dbutils.widgets.get("modo"),
    "data_referencia": dbutils.widgets.get("data_referencia").strip() or None,
    "raiz_arquivos": dbutils.widgets.get("raiz_arquivos").strip() or None,
}
batch_id = dbutils.widgets.get("batch_id").strip()
if batch_id:
    parametros["batch_id"] = batch_id

config = PipelineConfig(**parametros)
print("Ambiente:", config.ambiente)
print("Catálogo:", config.catalogo)
print("Batch:", config.batch_id)
print("Entrada planejada (ainda não validada):", config.caminho_entrada)



In [0]:
# CMD 4 — Criar/verificar os cinco schemas dev. Esperado: ESTRUTURA DO PROJETO: OK.
schemas = preparar_ambiente(spark, config)

print("ESTRUTURA DO PROJETO: OK")
print("Pacote Python: importado")
print("Configuração: validada")
print(f"Ambiente: {config.ambiente}")
print(f"Catálogo: {config.catalogo}")
for schema in schemas:
    print(f"Schema disponível: {schema}")
print("Próxima etapa: contratos, Volume e dados sintéticos.")



## Coleta desta revalidação

Após executar todos os CMDs, confira a mensagem final esperada. Se ocorrer erro, preserve o output e interrompa a sequência.

Exporte este notebook como **IPYNB com outputs**, abra o arquivo e confirme que os resultados estão presentes. Salve o arquivo antes de limpar outputs ou mudar a sessão. Faça commit/push na branch `chore/estrutura-inicial`, registre o SHA e envie o export para revisão antes de avançar ao próximo notebook.

Sugestão de commit: `docs: revalida 01_preparar_projeto com instrucoes de parametros`.

O commit sozinho não comprova que os outputs foram incluídos. A cópia pública do export será revisada para remover e-mails e caminhos pessoais, com manifesto e hashes, preservando a coleta anterior. Esta rodada só será registrada como validada após revisão do novo export.
